# Model Evaluation — Team11 Joel Puthenparambil

Evaluates all three model outputs using **BERTScore** (`xlm-roberta-base`).

Reference answers come from the course dataset Google Sheet. Each model's answers are compared against these.

Upload the dataset xlsx and the three result CSVs when prompted.

## Cell 1 — Install Libraries

This cell installs the packages needed for evaluation.

- `bert-score`: the library that computes BERTScore using contextual embeddings.
- `pandas`: used to load and merge the CSV and Excel files.
- `openpyxl`: required by pandas to read `.xlsx` files.

**Project Impact**

Without `bert-score` the evaluation cannot run. `openpyxl` is easy to forget — pandas silently fails to read Excel files without it.

In [ ]:
# Install evaluation library
!pip install bert-score pandas openpyxl

## Cell 2 — Imports

This cell imports the two libraries used throughout the notebook.

- `import pandas as pd`: loads CSV and Excel files into DataFrames.
- `from bert_score import score as bert_score`: imports the BERTScore scoring function.

**Project Impact**

All imports are placed here so a kernel restart won't cause `NameError` in later cells.

In [ ]:
# Imports
import pandas as pd
from bert_score import score as bert_score

## Load files

Upload:
- `dataset.xlsx` — the course Google Sheet downloaded as Excel (contains ground truth answers)
- `model1_inference.csv`
- `model2_finetuned.csv`
- `model3_rag.csv`

## Cell 3 — Upload Files

This cell opens a file picker to upload all required files into the Colab runtime.

- `files.upload()`: opens the browser file picker — select all four files at once.
- Upload `Austrian Tax Law Dataset.xlsx` (contains ground-truth answers), `model1_inference.csv`, `model2_finetuned.csv`, and `model3_rag.csv`.

**Project Impact**

All four files must be uploaded before any evaluation cell can run. The Excel file contains the gold-standard answers that all three models are scored against.

In [ ]:
# Upload files (Colab)
from google.colab import files
uploaded = files.upload()

## Cell 4 — Load Ground Truth from Excel

This cell reads the course dataset Excel file and extracts the gold-standard answers.

- `pd.read_excel('Austrian Tax Law Dataset.xlsx', sheet_name='Dataset')`: reads the `Dataset` sheet from the uploaded Excel file.
- `gt.columns.tolist()` and `len(gt)`: prints column names and row count to verify the file loaded correctly.

**Project Impact**

The ground-truth answers come from the course team's manually verified answers in the Google Sheet (downloaded as Excel). Using the wrong sheet or column would silently evaluate against incorrect references.

In [ ]:
# Load ground truth from the 'Dataset' sheet, column 'correct_answer'
gt = pd.read_excel('Austrian Tax Law Dataset.xlsx', sheet_name='Dataset')
print("Columns:", gt.columns.tolist())
print("Rows:", len(gt))
gt.head(2)

## Cell 5 — Extract and Clean Ground Truth

This cell selects only the relevant columns and removes rows without a reference answer.

- `ID_COL = 'id'` and `ANSWER_COL = 'correct_answer'`: defines which columns to use from the sheet.
- `gt[[ID_COL, ANSWER_COL]].rename(...)`: keeps only the ID and answer columns and renames `correct_answer` to `gold`.
- `gt.dropna(subset=['gold'])`: removes any rows where no reference answer exists.

**Project Impact**

Only rows with a `gold` answer can be used for evaluation. Keeping this step explicit makes it easy to spot if rows are unexpectedly dropped.

In [ ]:
# Column names confirmed from the actual file
ID_COL     = 'id'             # question ID
ANSWER_COL = 'correct_answer' # ground truth answer column

gt = gt[[ID_COL, ANSWER_COL]].rename(columns={ANSWER_COL: 'gold'})
gt = gt.dropna(subset=['gold'])  # drop rows with no reference answer
print(f"Ground truth rows: {len(gt)}")
gt.head(2)

## Cell 6 — Load Model Outputs & Merge with Ground Truth

This cell loads all three model result CSVs and joins them with the ground-truth answers on the `id` column.

- `pd.read_csv('model1_inference.csv')` etc.: loads each model's answers.
- `.rename(columns={'answer': 'model1'})`: renames the answer column so all three can coexist in one DataFrame.
- `df.merge(..., on='id', how='inner')`: keeps only rows present in both the ground truth and the model output.
- `df.fillna('')`: replaces any missing answers with empty strings to avoid errors in BERTScore.

**Project Impact**

The inner merge ensures evaluation only runs on questions that have both a gold answer and a model answer. The final DataFrame is the single source of truth for all three BERTScore computations.

In [ ]:
# Load model outputs and merge with ground truth on id
m1 = pd.read_csv('model1_inference.csv')
m2 = pd.read_csv('model2_finetuned.csv')
m3 = pd.read_csv('model3_rag.csv')

df = gt.copy()
df = df.merge(m1[['id','answer']].rename(columns={'answer':'model1'}), on='id', how='inner')
df = df.merge(m2[['id','answer']].rename(columns={'answer':'model2'}), on='id', how='inner')
df = df.merge(m3[['id','answer']].rename(columns={'answer':'model3'}), on='id', how='inner')
df = df.fillna('')

print(f"Questions with all answers: {len(df)}")
df.head(2)

## BERTScore — each model vs ground truth

## Cell 7 — Compute BERTScore for All Three Models

This cell scores each model's answers against the gold-standard answers using BERTScore.

- `refs = df['gold'].tolist()`: the list of reference answers used for all three comparisons.
- `bert_score(candidates, refs, model_type='xlm-roberta-base', lang='de')`: computes token-level precision, recall, and F1 using contextual embeddings from XLM-RoBERTa.
- `P.mean().item()`, `R.mean().item()`, `F1.mean().item()`: averages the per-question scores across all 643 questions.
- The function is called once per model (Model 1, 2, and 3) against the same `refs`.

**Project Impact**

BERTScore uses contextual embeddings instead of exact word matches, making it well-suited for German legal text where paraphrasing is common. XLM-RoBERTa covers German natively, giving more reliable scores than English-only models.

In [ ]:
refs = df['gold'].tolist()

# Model 1
print("Computing BERTScore: Model 1 vs ground truth ...")
P1, R1, F1_1 = bert_score(
    df['model1'].tolist(), refs,
    model_type='xlm-roberta-base', lang='de', verbose=False
)
print(f"  Precision: {P1.mean().item():.4f}  Recall: {R1.mean().item():.4f}  F1: {F1_1.mean().item():.4f}")

# Model 2
print("Computing BERTScore: Model 2 vs ground truth ...")
P2, R2, F1_2 = bert_score(
    df['model2'].tolist(), refs,
    model_type='xlm-roberta-base', lang='de', verbose=False
)
print(f"  Precision: {P2.mean().item():.4f}  Recall: {R2.mean().item():.4f}  F1: {F1_2.mean().item():.4f}")

# Model 3
print("Computing BERTScore: Model 3 vs ground truth ...")
P3, R3, F1_3 = bert_score(
    df['model3'].tolist(), refs,
    model_type='xlm-roberta-base', lang='de', verbose=False
)
print(f"  Precision: {P3.mean().item():.4f}  Recall: {R3.mean().item():.4f}  F1: {F1_3.mean().item():.4f}")

## Summary table

## Cell 8 — Build Summary Table & Save to CSV

This cell combines all three models' scores into a summary table and saves it.

- `pd.DataFrame([...])`: creates a table with one row per model and columns for Precision, Recall, and F1.
- `round(..., 4)`: rounds scores to 4 decimal places for readability.
- `summary.to_csv('evaluation_report.csv', index=False)`: saves the table to `evaluation_report.csv`.

**Project Impact**

Produces the final evaluation report that goes into `results/evaluation_report.csv`. This is the file referenced in the README results table.

In [ ]:
# Combine results and save
summary = pd.DataFrame([
    {
        'model': 'Model 1 (Mistral-7B inference)',
        'BERTScore_Precision': round(P1.mean().item(), 4),
        'BERTScore_Recall':    round(R1.mean().item(), 4),
        'BERTScore_F1':        round(F1_1.mean().item(), 4),
    },
    {
        'model': 'Model 2 (TinyLlama fine-tuned)',
        'BERTScore_Precision': round(P2.mean().item(), 4),
        'BERTScore_Recall':    round(R2.mean().item(), 4),
        'BERTScore_F1':        round(F1_2.mean().item(), 4),
    },
    {
        'model': 'Model 3 (RAG + Gemini)',
        'BERTScore_Precision': round(P3.mean().item(), 4),
        'BERTScore_Recall':    round(R3.mean().item(), 4),
        'BERTScore_F1':        round(F1_3.mean().item(), 4),
    },
])

summary.to_csv('evaluation_report.csv', index=False)
print("Saved evaluation_report.csv")
summary

## Cell 9 — Download Report

This cell downloads the evaluation report to your local machine.

- `files.download('evaluation_report.csv')`: triggers a browser download of the saved report.

**Project Impact**

Download immediately — Colab runtimes are temporary. This file is then copied to `results/evaluation_report.csv` in the submission repository.

In [ ]:
# Download the report
files.download('evaluation_report.csv')